# 2차 — 구조적 변형 스태킹 (IVF/DI 분리 + 기증난자 가중치)

## 배경
개별로는 전부 baseline보다 못했던 기법들:
- IVF/DI 분리(라우터) — 단독 손해
- 기증난자 weight=2x — 단독 손해 (거의 동일)

근데 팀원 제안대로 **이 둘을 baseline과 함께 블렌딩**했더니, 가벼운 설정(단일시드, n_estimators=200)
으로 빠르게 본 결과 +0.00044 개선이 나왔음. 노이즈(0.00011)의 4배 정도라 우연 치고는 크지만,
단일 시드라 확정 짓기엔 이름. **실제 robust params + 멀티시드로 정밀검증.**

## 메커니즘
개별 모델은 baseline과 상관관계가 0.989~0.997로 매우 높지만(건드리는 행이 전체의 2.5%/6.15%뿐이라
나머지는 거의 동일), **그 미세한 차이가 baseline과 다른 방향으로 틀리는 부분이 있다면** 블렌딩이
그 부분을 보완해줄 수 있음 — "개별로는 손해, 블렌딩하면 이득"이 이론적으로 가능한 상황.

## 누수 방지
가중치 탐색은 fold의 train 부분에서만 (nested CV) — 이전 블렌딩 노트북과 동일한 원칙.

---
**저장 위치:** `experiment_history/2차/2차_structural_stack.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
SEEDS = [1004, 42, 7]  # 멀티시드 (필요하면 늘려서 정밀검증)


## 1. 데이터 로드 + 전처리

In [3]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))

y = df_train[TARGET_COL].values.astype(int)
X = df_train.drop(columns=[TARGET_COL])

is_di = (train_raw["시술 유형"] == "DI").values
is_ivf = ~is_di
donor_egg_mask = (train_raw["난자 출처"] == "기증 제공").values

print(f"train shape: {X.shape}")
print(f"DI 비율: {is_di.mean():.2%}, 기증난자 비율: {donor_egg_mask.mean():.2%}")


train shape: (256351, 67)
DI 비율: 2.45%, 기증난자 비율: 6.15%


## 2. 시작 파라미터 로드

In [4]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")


파라미터 출처: ../lgbm_robust_best_params.json


## 3. 세 가지 변형의 OOF 생성 (시드별)

In [5]:
def make_oofs_for_seed(seed):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    folds = list(skf.split(X, y))

    oof_baseline = np.zeros(len(y))
    for tr_idx, val_idx in folds:
        m = LGBMClassifier(**base_params, random_state=seed, verbose=-1)
        m.fit(X.iloc[tr_idx], y[tr_idx])
        oof_baseline[val_idx] = m.predict_proba(X.iloc[val_idx])[:, 1]

    oof_routed = np.zeros(len(y))
    for sub_mask in [is_di, is_ivf]:
        X_sub, y_sub = X[sub_mask].reset_index(drop=True), y[sub_mask]
        skf_sub = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        oof_sub = np.zeros(len(y_sub))
        for tr_idx, val_idx in skf_sub.split(X_sub, y_sub):
            m = LGBMClassifier(**base_params, random_state=seed, verbose=-1)
            m.fit(X_sub.iloc[tr_idx], y_sub[tr_idx])
            oof_sub[val_idx] = m.predict_proba(X_sub.iloc[val_idx])[:, 1]
        oof_routed[sub_mask] = oof_sub

    oof_eggweight = np.zeros(len(y))
    for tr_idx, val_idx in folds:
        sw = np.ones(len(tr_idx))
        sw[donor_egg_mask[tr_idx]] = 2.0
        m = LGBMClassifier(**base_params, random_state=seed, verbose=-1)
        m.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=sw)
        oof_eggweight[val_idx] = m.predict_proba(X.iloc[val_idx])[:, 1]

    return oof_baseline, oof_routed, oof_eggweight


all_oofs = {"baseline": [], "routed": [], "eggweight": []}
for seed in SEEDS:
    t0 = time.time()
    ob, orr, oe = make_oofs_for_seed(seed)
    all_oofs["baseline"].append(ob)
    all_oofs["routed"].append(orr)
    all_oofs["eggweight"].append(oe)
    print(f"시드 {seed} 완료 ({time.time()-t0:.0f}s) — baseline={roc_auc_score(y,ob):.5f} "
          f"routed={roc_auc_score(y,orr):.5f} eggweight={roc_auc_score(y,oe):.5f}")

# 시드 평균(배깅) OOF
oof_baseline_bagged = np.mean(all_oofs["baseline"], axis=0)
oof_routed_bagged = np.mean(all_oofs["routed"], axis=0)
oof_eggweight_bagged = np.mean(all_oofs["eggweight"], axis=0)

print()
print(f"{len(SEEDS)}시드 배깅 baseline:  {roc_auc_score(y, oof_baseline_bagged):.5f}")
print(f"{len(SEEDS)}시드 배깅 routed:    {roc_auc_score(y, oof_routed_bagged):.5f}")
print(f"{len(SEEDS)}시드 배깅 eggweight: {roc_auc_score(y, oof_eggweight_bagged):.5f}")


시드 1004 완료 (55s) — baseline=0.73985 routed=0.73977 eggweight=0.73970
시드 42 완료 (51s) — baseline=0.73997 routed=0.73995 eggweight=0.73984
시드 7 완료 (51s) — baseline=0.73992 routed=0.74003 eggweight=0.74001

3시드 배깅 baseline:  0.74021
3시드 배깅 routed:    0.74022
3시드 배깅 eggweight: 0.74018


## 4. Nested CV 블렌딩 (배깅된 OOF들끼리)

In [6]:
def softmax_weights(theta):
    e = np.exp(theta - np.max(theta))
    return e / e.sum()


def optimize_weights(mat, y_sub, n_models, n_restarts=3, seed=0):
    rng = np.random.RandomState(seed)
    best_w, best_auc = None, -1.0
    inits = [np.zeros(n_models)] + [rng.normal(0, 1, n_models) for _ in range(n_restarts - 1)]
    for theta0 in inits:
        res = minimize(lambda t: -roc_auc_score(y_sub, mat @ softmax_weights(t)), theta0,
                        method="Nelder-Mead", options={"maxiter": 3000})
        w = softmax_weights(res.x)
        auc = roc_auc_score(y_sub, mat @ w)
        if auc > best_auc:
            best_auc, best_w = auc, w
    return best_w


mat = np.column_stack([oof_baseline_bagged, oof_routed_bagged, oof_eggweight_bagged])

blend_skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=999)
blended_oof = np.zeros(len(y))
fold_weights = []
for tr_idx, val_idx in blend_skf.split(X, y):
    w = optimize_weights(mat[tr_idx], y[tr_idx], 3)
    blended_oof[val_idx] = mat[val_idx] @ w
    fold_weights.append(w)

blend_auc = roc_auc_score(y, blended_oof)
print(f"baseline 단독({len(SEEDS)}시드 배깅): {roc_auc_score(y, oof_baseline_bagged):.5f}")
print(f"Nested CV 블렌딩(3개 구조 변형):      {blend_auc:.5f}")
print(f"차이: {blend_auc - roc_auc_score(y, oof_baseline_bagged):+.5f}")
print(f"평균 가중치: baseline={np.mean([w[0] for w in fold_weights]):.3f}, "
      f"routed={np.mean([w[1] for w in fold_weights]):.3f}, "
      f"eggweight={np.mean([w[2] for w in fold_weights]):.3f}")
print()
print("비교 기준:")
print("  10시드 배깅 baseline: 0.74036 (기존) / 0.74037 (견고탐색)")
print("  팀 블렌딩(가중평균, 2개 모델): 0.74071")


baseline 단독(3시드 배깅): 0.74021
Nested CV 블렌딩(3개 구조 변형):      0.74031
차이: +0.00010
평균 가중치: baseline=0.187, routed=0.489, eggweight=0.324

비교 기준:
  10시드 배깅 baseline: 0.74036 (기존) / 0.74037 (견고탐색)
  팀 블렌딩(가중평균, 2개 모델): 0.74071


## 5. 해석 가이드

- 블렌딩이 baseline(같은 시드수 기준) 대비 노이즈(0.00011)의 3배 이상 개선되면 — **진짜 신호**.
  `SEEDS`를 10개로 늘려서(`[1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]`) 한 번 더 확정 검증
  추천 (시간이 꽤 걸릴 수 있음 — 변형이 3종류라 기존 노트북들보다 3배 느림)
- 개선폭이 노이즈 수준이면 — 단일시드 빠른 테스트에서 본 +0.00044가 우연이었다는 뜻
- 어느 쪽이든, 이게 우리가 만들 수 있는 "구조적 변형 스태킹"의 거의 최선일 가능성이 높음 — 더
  추가할 변형이 마땅치 않다면(이미 시도한 것들 중 위 두 개가 그나마 덜 나빴던 것들) 여기서 결론
